# Parte 3 — Text-to-Image (Colab T4)

**Owner:** Santiago Diaz

Genera una imagen por cada nombre de dinosaurio (10 en total) con el modelo de difusión `amused/amused-512` — liviano, corre en T4 sin problema.

**Cómo correr:**
1. Subir este notebook a Colab.
2. Runtime → Change runtime type → **T4 GPU**.
3. Subir `top10_names.json` y `descriptions.json` (los entrega Juan Camilo) o pegarlos en la celda de datos.
4. Ejecutar todo. Las imágenes se guardan en `images/` y se descargan como ZIP.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors

In [ ]:
import torch, os, json
from pathlib import Path
from diffusers import AmusedPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
assert device == 'cuda', 'Activa el runtime T4 GPU antes de continuar'

## 1. Datos: nombres + descripciones

Si subiste los archivos al Colab, se leen directo. Si no, pega aquí los 10 nombres y descripciones.

In [ ]:
# Opción A: leer archivos subidos
if Path('descriptions.json').exists():
    items = json.loads(Path('descriptions.json').read_text())
else:
    # Opción B: pegar manualmente (placeholder de ejemplo)
    items = [
        {'name': 'mangosaurus', 'description': 'Saurópodo herbívoro de cuello largo y cresta dorsal.'},
        {'name': 'velocirex',   'description': 'Terópodo bípedo de patas largas y plumaje oscuro.'},
        # ... completar con los 10 reales
    ]

print(f'items: {len(items)}')
for it in items[:3]:
    print(' -', it['name'], '|', it['description'][:60], '...')

## 2. Cargar pipeline de difusión

In [ ]:
pipe = AmusedPipeline.from_pretrained(
    'amused/amused-512',
    torch_dtype=torch.float16,
).to(device)
pipe.set_progress_bar_config(disable=True)
print('pipeline listo')

## 3. Generar las 10 imágenes

In [ ]:
PROMPT_TEMPLATE = (
    'A paleo-illustration of a dinosaur named {name}. '
    '{description} '
    'Realistic, museum diorama style, neutral background, full body visible.'
)
NEGATIVE = 'cartoon, low quality, blurry, watermark, text, deformed, multiple heads'

Path('images').mkdir(exist_ok=True)
manifest = []

for i, it in enumerate(items, 1):
    prompt = PROMPT_TEMPLATE.format(**it)
    print(f'[{i}/{len(items)}] {it["name"]}')
    image = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        height=512, width=512,
        num_inference_steps=12,
        guidance_scale=7.5,
    ).images[0]
    out_path = f'images/{it["name"]}.png'
    image.save(out_path)
    manifest.append({'name': it['name'], 'image': out_path, 'prompt': prompt})

Path('images/manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print('\nlistas las 10 imágenes en images/')

## 4. Descargar como ZIP (o subir directo a S3)

In [ ]:
# Opción A: descargar ZIP al laptop
import shutil
shutil.make_archive('dino_images', 'zip', 'images')
from google.colab import files
files.download('dino_images.zip')

In [ ]:
# Opción B: subir directo a S3 (requiere credenciales AWS)
# !pip install -q awscli
# import os; os.environ['AWS_ACCESS_KEY_ID']='...'; os.environ['AWS_SECRET_ACCESS_KEY']='...'
# !aws s3 sync images/ s3://$S3_BUCKET/images/ --acl public-read

## 5. (Opcional) Exponer pipeline vía ngrok para `/new-dinosaur`

Para que el botón "Nuevo Dinosaurio" del sitio web pueda generar imágenes en vivo, la difusión debe quedar disponible como endpoint HTTP. Una opción sencilla es FastAPI + pyngrok dentro del mismo Colab:

```python
!pip install -q fastapi uvicorn pyngrok nest-asyncio
from fastapi import FastAPI; from pydantic import BaseModel
import nest_asyncio, uvicorn, threading
from pyngrok import ngrok

app = FastAPI()
class Req(BaseModel): name: str; description: str

@app.post('/image')
def make(req: Req):
    img = pipe(prompt=PROMPT_TEMPLATE.format(**req.dict()), height=512, width=512,
               num_inference_steps=12).images[0]
    path = f'images/{req.name}.png'; img.save(path)
    return {'image_url': path}

ngrok.set_auth_token('TU_TOKEN_NGROK')
url = ngrok.connect(8000); print('público:', url)
nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000), daemon=True).start()
```

Pasarle la URL pública a **Alan** para configurar `DIFFUSION_NGROK_URL` en la Lambda.